# 01 — EDA: Home Credit Default Risk

**Fuente de datos declarada en la primera celda de salida.** Si dice `synthetic`,
los hallazgos describen el generador sintético (útil para validar el pipeline);
las conclusiones de negocio solo valen cuando la fuente sea `kaggle`.


In [1]:
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src import config
from src.data import load_data, load_secondary
from src.aggregates import merge_aggregates
from src.features import add_domain_features

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
config.REPORTS_DIR.mkdir(exist_ok=True)

app, source = load_data()
df = merge_aggregates(app, load_secondary(app, source))
df = add_domain_features(df)
print(f"FUENTE: {source} | filas: {len(df):,} | columnas: {df.shape[1]}")
print(f"Tasa de default: {df[config.TARGET].mean():.3%}")

FUENTE: kaggle | filas: 307,511 | columnas: 46
Tasa de default: 8.073%


## 1. Desbalance del target

La tasa de default (~7–8%) define todo lo demás: métricas (AUC/KS en vez de
accuracy), validación estratificada, y umbral de decisión por costos en vez
de `class_weight` (rebalancear la pérdida de entrenamiento distorsiona
`predict_proba` sin mejorar el ranking — ver decisiones técnicas en CLAUDE.md).

In [2]:
counts = df[config.TARGET].value_counts()
fig, ax = plt.subplots(figsize=(5,3))
ax.bar(["Pagó (0)", "Default (1)"], counts.values, color=["#2563EB", "#DC2626"])
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}\n({v/len(df):.1%})", ha="center", va="bottom")
ax.set_title(f"Desbalance del target (fuente: {source})"); ax.set_ylabel("solicitudes")
fig.tight_layout(); fig.savefig(config.REPORTS_DIR / "eda_target_balance.png")
print("→ reports/eda_target_balance.png")

→ reports/eda_target_balance.png


## 2. Nulos

`EXT_SOURCE_1` (~44%) y `EXT_SOURCE_3` (~17%) concentran los faltantes. Decisión:
imputación por mediana **dentro del Pipeline** + feature `EXT_SOURCES_NULLS`
(la ausencia de score de buró es señal en sí misma).

In [3]:
nulls = df.isna().mean().sort_values(ascending=False)
print(nulls[nulls > 0].to_string(float_format="{:.1%}".format))

EXT_SOURCE_1                    56.4%
EXT_SOURCE_3                    19.8%
EMPLOYED_AGE_RATIO              18.0%
DAYS_EMPLOYED                   18.0%
EMPLOYED_YEARS                  18.0%
BUREAU_OVERDUE_MAX              14.3%
BUREAU_DAYS_CREDIT_MEAN         14.3%
BUREAU_DEBT_CREDIT_RATIO        14.3%
BUREAU_CREDIT_SUM               14.3%
BUREAU_DEBT_SUM                 14.3%
PREV_AMT_APPLICATION_MEAN        5.4%
PREV_REFUSED_RATE                5.4%
PREV_DAYS_DECISION_MEAN          5.4%
PREV_CREDIT_APPLICATION_RATIO    5.4%
EXT_SOURCE_2                     0.2%
AMT_GOODS_PRICE                  0.1%
GOODS_CREDIT_RATIO               0.1%
EXT_SOURCES_MIN                  0.1%
EXT_SOURCES_MEAN                 0.1%
AMT_ANNUITY                      0.0%
ANNUITY_INCOME_RATIO             0.0%
CREDIT_TERM                      0.0%


## 3. Poder discriminante de las variables clave

Comparamos distribuciones por clase para los scores externos y ratios de dominio.

In [4]:
key_feats = ["EXT_SOURCES_MEAN", "CREDIT_INCOME_RATIO", "EMPLOYED_YEARS", "AGE_YEARS"]
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, col in zip(axes.ravel(), key_feats):
    for t, color, label in [(0, "#2563EB", "pagó"), (1, "#DC2626", "default")]:
        vals = df.loc[df[config.TARGET] == t, col].dropna()
        ax.hist(vals, bins=40, density=True, alpha=0.55, color=color, label=label)
    ax.set_title(col, fontsize=10); ax.legend(fontsize=8)
fig.suptitle(f"Distribuciones por clase (fuente: {source})")
fig.tight_layout(); fig.savefig(config.REPORTS_DIR / "eda_class_distributions.png")
print("→ reports/eda_class_distributions.png")

→ reports/eda_class_distributions.png


## 4. Tasa de default por deciles (univariate lift)

La lectura de riesgo estándar: ¿ordena la variable a los clientes por riesgo?

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, col in zip(axes, ["EXT_SOURCES_MEAN", "CREDIT_INCOME_RATIO", "BUREAU_OVERDUE_ANY"]):
    if df[col].nunique() > 10:
        binned = pd.qcut(df[col], 10, duplicates="drop")
        rate = df.groupby(binned, observed=True)[config.TARGET].mean()
        ax.plot(range(len(rate)), rate.values, marker="o", color="#2563EB")
        ax.set_xlabel("decil")
    else:
        rate = df.groupby(col)[config.TARGET].mean()
        ax.bar(rate.index.astype(str), rate.values, color="#2563EB")
    ax.axhline(df[config.TARGET].mean(), ls="--", c="gray", lw=1)
    ax.set_title(col, fontsize=10); ax.set_ylabel("tasa default")
fig.tight_layout(); fig.savefig(config.REPORTS_DIR / "eda_default_by_decile.png")
print("→ reports/eda_default_by_decile.png")

→ reports/eda_default_by_decile.png


## 5. Historial multi-tabla

Señales de las agregaciones de buró y solicitudes previas.

In [6]:
agg_cols = ["BUREAU_LOAN_COUNT", "BUREAU_OVERDUE_ANY", "PREV_REFUSED_COUNT", "PREV_REFUSED_RATE"]
summary = df.groupby(config.TARGET)[agg_cols].mean().T
summary.columns = ["pagó (0)", "default (1)"]
summary["lift"] = summary["default (1)"] / summary["pagó (0)"]
print(summary.round(3).to_string())

                    pagó (0)  default (1)   lift
BUREAU_LOAN_COUNT      4.778        4.613  0.965
BUREAU_OVERDUE_ANY     0.010        0.022  2.152
PREV_REFUSED_COUNT     0.764        1.186  1.552
PREV_REFUSED_RATE      0.107        0.159  1.490


## Hallazgos (fuente: kaggle, 307,511 filas, default_rate=8.07%)

1. **Desbalance ~92/8** -> AUC/KS + class weights + CV estratificado (confirmado con datos reales).
2. **EXT_SOURCE_* son las variables mas discriminantes** -- consistente con lo
   reportado en el problema real de Home Credit; su nulidad es informativa
   (EXT_SOURCE_1 falta en 56.4% de los casos reales, mas que en el sintetico).
3. **CREDIT_INCOME_RATIO ordena el riesgo monotonicamente** -- el apalancamiento
   importa, y mas en clientes con score de buro bajo (interaccion).
4. **El historial multi-tabla agrega senal, pero con matices reales**:
   `PREV_REFUSED_COUNT`/`PREV_REFUSED_RATE` si muestran lift claro (1.55/1.49).
   `BUREAU_OVERDUE_ANY` tiene lift alto (2.15) pero base rate muy baja (~1-2%
   de las cuentas), asi que aporta poca cobertura. `BUREAU_LOAN_COUNT` **no**
   ordena en la direccion esperada en real (lift 0.97, practicamente plano) --
   a diferencia del sintetico (1.20). Se deja en el modelo porque no dana
   (el arbol puede ignorarlo), pero no se le atribuye poder predictivo real.
5. **Nulos concentrados en fuentes externas e historial de buro/previas**
   -> imputacion dentro del Pipeline (paridad train/serve) y contador de
   nulos como feature.
6. **DAYS_EMPLOYED == 365243 (sentinel de nulo, ~18% de las filas)**: resuelto
   en `src/data.py::_clean_sentinels` -> se convierte a NaN antes de feature
   engineering, en vez de colapsar silenciosamente a `EMPLOYED_YEARS=0` (bug
   corregido; ver `tests/test_pipeline.py::test_days_employed_sentinel_becomes_nan`).
   Impacto en metricas: marginal y en ambas direcciones (holdout AUC
   0.7730->0.7739, KS 0.4083->0.4112) -- consistente con una correccion de
   calidad de datos, no con una mejora artificial.

**Anomalias de EDA, actualizado 2026-07-21:**
- `AMT_INCOME_TOTAL` tenia un outlier extremo (117,000,000 -- ~130x el
  percentil 99.9 de 900,000), probablemente error de captura. **Corregido:**
  `src/data.py::_cap_income_outliers` lo recorta a un tope fijo de 1,000,000
  (constante, no calculada del split train/test, mismo criterio anti-leakage
  que el sentinel de DAYS_EMPLOYED). Afecta 250/307,511 filas (0.08%); impacto
  en metricas fue ruido, no una mejora artificial. Test:
  `test_amt_income_outlier_is_capped_not_dropped`.
- `CNT_CHILDREN` tiene 2 filas con valor 19 (salto abrupto desde el maximo
  tipico de 14), posible error de captura. **Decision: documentado, sin
  corregir** -- volumen insignificante (2/307,511) y, a diferencia del ingreso,
  no hay evidencia tan clara de que sea un error vs. una familia numerosa real.

**Resuelto (ya no pendiente):** `ORGANIZATION_TYPE` (58 categorias) se agrego
a `config.CATEGORICAL_COLS`; el `OneHotEncoder(min_frequency=0.01)` existente
ya agrupaba categorias raras, sin codigo nuevo. Como grupo, queda 7a en
importancia SHAP.

**Pendiente (fuera de alcance actual, no bloquea):** estabilidad temporal si
se usa `DAYS_DECISION` como eje.